# Dataset Analysis Notebook

This notebook analyzes the poultry pose dataset:
- Dataset statistics
- Class distribution
- Keypoint completeness
- Annotation visualization

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter

## 1. Load Dataset

In [ ]:
# Find pose label project files
project_files = list(Path('..').rglob('*.poselabel'))
print(f"Found {len(project_files)} annotation project(s):")
for f in project_files:
    print(f"  - {f}")

# Load the first project
project_data = None
if project_files:
    with open(project_files[0], 'r') as f:
        project_data = json.load(f)
    print(f"\nLoaded: {project_files[0].name}")

## 2. Dataset Statistics

In [ ]:
if project_data:
    frames = project_data.get('frames', {})
    schema = project_data.get('schema', {})
    
    # Basic stats
    total_frames = len(frames)
    annotated_frames = sum(1 for f in frames.values() if f.get('instances'))
    
    # Instance counts
    total_instances = sum(len(f.get('instances', [])) for f in frames.values())
    
    print("Dataset Statistics:")
    print(f"  Total frames: {total_frames}")
    print(f"  Annotated frames: {annotated_frames} ({100*annotated_frames/total_frames:.1f}%)")
    print(f"  Total instances: {total_instances}")
    print(f"  Avg instances/frame: {total_instances/annotated_frames:.1f}" if annotated_frames else "N/A")
    
    # Schema info
    keypoints = schema.get('keypoints', [])
    print(f"\nSchema: {schema.get('name', 'Unknown')}")
    print(f"  Classes: {list(schema.get('classes', {}).values())}")
    print(f"  Keypoints: {len(keypoints)}")
    for kp in keypoints:
        print(f"    - {kp.get('name')}")

## 3. Keypoint Completeness Analysis

In [ ]:
if project_data:
    keypoints = schema.get('keypoints', [])
    kp_names = [kp.get('name') for kp in keypoints]
    
    # Count keypoint visibility across all instances
    visibility_counts = {name: {'visible': 0, 'occluded': 0, 'unlabeled': 0} for name in kp_names}
    
    for frame in frames.values():
        for instance in frame.get('instances', []):
            for kp_name, kp_data in instance.get('keypoints', {}).items():
                vis = kp_data.get('visibility', 0)
                if kp_name in visibility_counts:
                    if vis == 2:
                        visibility_counts[kp_name]['visible'] += 1
                    elif vis == 1:
                        visibility_counts[kp_name]['occluded'] += 1
                    else:
                        visibility_counts[kp_name]['unlabeled'] += 1
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(kp_names))
    width = 0.25
    
    visible = [visibility_counts[n]['visible'] for n in kp_names]
    occluded = [visibility_counts[n]['occluded'] for n in kp_names]
    unlabeled = [visibility_counts[n]['unlabeled'] for n in kp_names]
    
    ax.bar(x - width, visible, width, label='Visible', color='green')
    ax.bar(x, occluded, width, label='Occluded', color='orange')
    ax.bar(x + width, unlabeled, width, label='Unlabeled', color='gray')
    
    ax.set_xlabel('Keypoint')
    ax.set_ylabel('Count')
    ax.set_title('Keypoint Annotation Completeness')
    ax.set_xticks(x)
    ax.set_xticklabels(kp_names, rotation=45, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Sample Visualization

In [ ]:
def visualize_annotation(image_path, frame_data, schema):
    """Visualize annotations on an image."""
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    
    h, w = img.shape[:2]
    keypoints = {kp['name']: kp for kp in schema.get('keypoints', [])}
    skeleton = schema.get('skeleton', [])
    kp_names = [kp['name'] for kp in schema.get('keypoints', [])]
    
    for instance in frame_data.get('instances', []):
        # Draw bounding box
        bbox = instance.get('bbox', {})
        if bbox:
            x1 = int((bbox['x_center'] - bbox['width']/2) * w)
            y1 = int((bbox['y_center'] - bbox['height']/2) * h)
            x2 = int((bbox['x_center'] + bbox['width']/2) * w)
            y2 = int((bbox['y_center'] + bbox['height']/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Draw keypoints
        kp_coords = {}
        for kp_name, kp_data in instance.get('keypoints', {}).items():
            vis = kp_data.get('visibility', 0)
            if vis > 0:
                x = int(kp_data['x'] * w)
                y = int(kp_data['y'] * h)
                kp_coords[kp_name] = (x, y)
                
                color = (0, 255, 0) if vis == 2 else (0, 165, 255)
                cv2.circle(img, (x, y), 5, color, -1)
        
        # Draw skeleton
        for s1, s2 in skeleton:
            n1, n2 = kp_names[s1], kp_names[s2]
            if n1 in kp_coords and n2 in kp_coords:
                cv2.line(img, kp_coords[n1], kp_coords[n2], (255, 0, 0), 2)
    
    return img

if project_data:
    # Find annotated frames
    annotated = [(k, v) for k, v in frames.items() if v.get('instances')]
    
    if annotated:
        fig, axes = plt.subplots(1, min(3, len(annotated)), figsize=(15, 5))
        if len(annotated) == 1:
            axes = [axes]
        
        for i, (img_path, frame_data) in enumerate(annotated[:3]):
            vis = visualize_annotation(img_path, frame_data, schema)
            if vis is not None:
                axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
                axes[i].set_title(f"Frame {i+1}")
                axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No annotated frames found")

## 5. Export Statistics

In [ ]:
if project_data:
    # Calculate completeness percentage
    total_possible_kps = total_instances * len(kp_names)
    total_labeled = sum(sum(c.values()) - c['unlabeled'] for c in visibility_counts.values())
    completeness = 100 * total_labeled / total_possible_kps if total_possible_kps else 0
    
    print("\n" + "="*50)
    print("Dataset Summary")
    print("="*50)
    print(f"Total frames: {total_frames}")
    print(f"Annotated frames: {annotated_frames}")
    print(f"Total instances: {total_instances}")
    print(f"Keypoint completeness: {completeness:.1f}%")
    print(f"Schema: {schema.get('name')}")